In [1]:
%pip install pyYAML 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import os 
import yaml

In [19]:
filenames=[]
for f in os.listdir('data/t20s'):
    filenames.append(os.path.join('data/t20s',f))

In [20]:
filenames[0:5]


['data/t20s\\1001349.yaml',
 'data/t20s\\1001351.yaml',
 'data/t20s\\1001353.yaml',
 'data/t20s\\1004729.yaml',
 'data/t20s\\1007655.yaml']

In [21]:

with open(filenames[0], "r", encoding="utf-8") as f:
    match = yaml.safe_load(f)

print(match["innings"][0])

{'1st innings': {'team': 'Australia', 'deliveries': [{0.1: {'batsman': 'AJ Finch', 'bowler': 'SL Malinga', 'non_striker': 'M Klinger', 'runs': {'batsman': 0, 'extras': 0, 'total': 0}}}, {0.2: {'batsman': 'AJ Finch', 'bowler': 'SL Malinga', 'non_striker': 'M Klinger', 'runs': {'batsman': 0, 'extras': 0, 'total': 0}}}, {0.3: {'batsman': 'AJ Finch', 'bowler': 'SL Malinga', 'non_striker': 'M Klinger', 'runs': {'batsman': 1, 'extras': 0, 'total': 1}}}, {0.4: {'batsman': 'M Klinger', 'bowler': 'SL Malinga', 'non_striker': 'AJ Finch', 'runs': {'batsman': 2, 'extras': 0, 'total': 2}}}, {0.5: {'batsman': 'M Klinger', 'bowler': 'SL Malinga', 'non_striker': 'AJ Finch', 'runs': {'batsman': 0, 'extras': 0, 'total': 0}}}, {0.6: {'batsman': 'M Klinger', 'bowler': 'SL Malinga', 'non_striker': 'AJ Finch', 'runs': {'batsman': 3, 'extras': 0, 'total': 3}}}, {1.1: {'batsman': 'M Klinger', 'bowler': 'KMDN Kulasekara', 'non_striker': 'AJ Finch', 'runs': {'batsman': 0, 'extras': 0, 'total': 0}}}, {1.2: {'bat

In [22]:
from yaml import safe_load
import pandas as pd
from tqdm import tqdm

def parse_yaml_to_rows(file, match_id):
    rows = []
    with open(file, 'r') as f:
        data = safe_load(f)

    info = data.get('info', {})
    
    # Extract match-level fields you want to carry into every row
    match_meta = {
        'match_id'       : match_id,
        'city'           : info.get('city'),
        'venue'          : info.get('venue'),
        'date'           : info.get('dates', [None])[0],
        'match_type'     : info.get('match_type'),
        'gender'         : info.get('gender'),
        'teams'          : str(info.get('teams', [])),
        'toss_winner'    : info.get('toss', {}).get('winner'),
        'toss_decision'  : info.get('toss', {}).get('decision'),
        'match_winner'   : info.get('outcome', {}).get('winner'),
        'win_by_runs'    : info.get('outcome', {}).get('by', {}).get('runs'),
        'win_by_wickets' : info.get('outcome', {}).get('by', {}).get('wickets'),
        'player_of_match': str(info.get('player_of_match', [])),
    }

    for innings in data.get('innings', []):
        for innings_name, innings_data in innings.items():
            team = innings_data.get('team')
            for delivery in innings_data.get('deliveries', []):
                for over_ball, details in delivery.items():
                    row = {
                        **match_meta,                          # all match-level columns
                        'innings'      : innings_name,
                        'team'         : team,
                        'over_ball'    : over_ball,
                        'batsman'      : details.get('batsman'),
                        'bowler'       : details.get('bowler'),
                        'non_striker'  : details.get('non_striker'),
                        'runs_batsman' : details['runs']['batsman'],
                        'runs_extras'  : details['runs']['extras'],
                        'runs_total'   : details['runs']['total'],
                        # extras breakdown
                        'wides'        : details.get('extras', {}).get('wides', 0),
                        'byes'         : details.get('extras', {}).get('byes', 0),
                        'legbyes'      : details.get('extras', {}).get('legbyes', 0),
                        'noballs'      : details.get('extras', {}).get('noballs', 0),
                        # wicket info
                        'wicket_kind'  : details.get('wicket', {}).get('kind'),
                        'player_out'   : details.get('wicket', {}).get('player_out'),
                        'fielders'     : str(details.get('wicket', {}).get('fielders', [])),
                    }
                    rows.append(row)
    return rows


# --- Main loop ---
all_rows = []

for counter, file in enumerate(tqdm(filenames), start=1):
    try:
        all_rows.extend(parse_yaml_to_rows(file, match_id=counter))
    except Exception as e:
        print(f"Skipped {file} due to error: {e}")

final_df = pd.DataFrame(all_rows)
final_df

 38%|███▊      | 2016/5342 [10:53<21:03,  2.63it/s]

Skipped data/t20s\1383095.yaml due to error: 'list' object has no attribute 'get'


 61%|██████    | 3269/5342 [17:40<09:10,  3.77it/s]

Skipped data/t20s\1459387.yaml due to error: 'list' object has no attribute 'get'


 67%|██████▋   | 3603/5342 [19:23<05:54,  4.91it/s]

...

 87%|████████▋ | 4669/5342 [24:59<03:18,  3.39it/s]

Skipped data/t20s\1535418.yaml due to error: 'list' object has no attribute 'get'


100%|██████████| 5342/5342 [28:40<00:00,  3.10it/s]

Skipped data/t20s\README.txt due to error: 'str' object has no attribute 'get'


,match_id,city,venue,date,match_type,gender,teams,toss_winner,toss_decision,match_winner,...,runs_batsman,runs_extras,runs_total,wides,byes,legbyes,noballs,wicket_kind,player_out,fielders
0,1,NaN,Melbourne Cricket Ground,2017-02-17,T20,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,...,0,0,0,0,0,0,0,NaN,NaN,[]
1,1,NaN,Melbourne Cricket Ground,2017-02-17,T20,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,...,0,0,0,0,0,0,0,NaN,NaN,[]
2,1,NaN,Melbourne Cricket Ground,2017-02-17,T20,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,...,1,0,1,0,0,0,0,NaN,NaN,[]
3,1,NaN,Melbourne Cricket Ground,2017-02-17,T20,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,...,2,0,2,0,0,0,0,NaN,NaN,[]
4,1,NaN,Melbourne Cricket Ground,2017-02-17,T20,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,...,0,0,0,0,0,0,0,NaN,NaN,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1206790,5341,Colombo,R Premadasa Stadium,2016-09-09,T20,male,"['Sri Lanka', 'Australia']",Sri Lanka,bat,Australia,...,1,0,1,0,0,0,0,NaN,NaN,[]
1206791,5341,Colombo,R Premadasa Stadium,2016-09-09,T20,male,"['Sri Lanka', 'Australia']",Sri Lanka,bat,Australia,...,3,0,3,0,0,0,0,NaN,NaN,[]
1206792,5341,Colombo,R Premadasa Stadium,2016-09-09,T20,male,"['Sri Lanka', 'Australia']",Sri Lanka,bat,Australia,...,0,0,0,0,0,0,0,NaN,NaN,[]
1206793,5341,Colombo,R Premadasa Stadium,2016-09-09,T20,male,"['Sri Lanka', 'Australia']",Sri Lanka,bat,Australia,...,0,0,0,0,0,0,0,NaN,NaN,[]


In [23]:
df_copy=final_df.copy()
df_copy.head()

,match_id,city,venue,date,match_type,gender,teams,toss_winner,toss_decision,match_winner,...,runs_batsman,runs_extras,runs_total,wides,byes,legbyes,noballs,wicket_kind,player_out,fielders
0,1,NaN,Melbourne Cricket Ground,2017-02-17,T20,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,...,0,0,0,0,0,0,0,NaN,NaN,[]
1,1,NaN,Melbourne Cricket Ground,2017-02-17,T20,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,...,0,0,0,0,0,0,0,NaN,NaN,[]
2,1,NaN,Melbourne Cricket Ground,2017-02-17,T20,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,...,1,0,1,0,0,0,0,NaN,NaN,[]
3,1,NaN,Melbourne Cricket Ground,2017-02-17,T20,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,...,2,0,2,0,0,0,0,NaN,NaN,[]
4,1,NaN,Melbourne Cricket Ground,2017-02-17,T20,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,...,0,0,0,0,0,0,0,NaN,NaN,[]


In [24]:
df_copy=df_copy.drop(['match_type','date','fielders'],axis=1)


In [25]:
df_copy = df_copy[df_copy['gender'] != 'female']

In [26]:
df_copy.head()

,match_id,city,venue,gender,teams,toss_winner,toss_decision,match_winner,win_by_runs,win_by_wickets,...,non_striker,runs_batsman,runs_extras,runs_total,wides,byes,legbyes,noballs,wicket_kind,player_out
0,1,NaN,Melbourne Cricket Ground,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,NaN,5.0,...,M Klinger,0,0,0,0,0,0,0,NaN,NaN
1,1,NaN,Melbourne Cricket Ground,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,NaN,5.0,...,M Klinger,0,0,0,0,0,0,0,NaN,NaN
2,1,NaN,Melbourne Cricket Ground,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,NaN,5.0,...,M Klinger,1,0,1,0,0,0,0,NaN,NaN
3,1,NaN,Melbourne Cricket Ground,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,NaN,5.0,...,AJ Finch,2,0,2,0,0,0,0,NaN,NaN
4,1,NaN,Melbourne Cricket Ground,male,"['Australia', 'Sri Lanka']",Sri Lanka,field,Sri Lanka,NaN,5.0,...,AJ Finch,0,0,0,0,0,0,0,NaN,NaN


In [27]:
df_copy.to_csv('cric_data.csv',index=False)

In [28]:
df_copy=pd.read_csv('cric_data.csv')

In [29]:
df_copy=df_copy.drop(['runs_batsman','runs_extras','wides','byes','legbyes','noballs'],axis=1)

In [30]:
df_copy=df_copy.drop(['win_by_runs','win_by_wickets'],axis=1)

In [31]:
df_copy=df_copy.drop(['batsman','bowler','non_striker','match_winner','player_of_match'],axis=1)


In [32]:
df_copy=df_copy.drop(['toss_winner','toss_decision','innings'],axis=1)


In [33]:
df_copy.head()

,match_id,city,venue,gender,teams,team,over_ball,runs_total,wicket_kind,player_out
0,1,NaN,Melbourne Cricket Ground,male,"['Australia', 'Sri Lanka']",Australia,0.1,0,NaN,NaN
1,1,NaN,Melbourne Cricket Ground,male,"['Australia', 'Sri Lanka']",Australia,0.2,0,NaN,NaN
2,1,NaN,Melbourne Cricket Ground,male,"['Australia', 'Sri Lanka']",Australia,0.3,1,NaN,NaN
3,1,NaN,Melbourne Cricket Ground,male,"['Australia', 'Sri Lanka']",Australia,0.4,2,NaN,NaN
4,1,NaN,Melbourne Cricket Ground,male,"['Australia', 'Sri Lanka']",Australia,0.5,0,NaN,NaN


In [34]:
import ast

df_copy['teams'] = df_copy['teams'].apply(ast.literal_eval)

In [35]:
df_copy['bowling_team'] = df_copy.apply(
    lambda row: [t for t in row['teams'] if t != row['team']][0], axis=1
)
df_copy.head()
df_copy.shape

(758141, 11)

In [36]:
df_copy['batting_team'] = df_copy.apply(
    lambda row: [t for t in row['teams'] if t == row['team']][0], axis=1
)
df_copy.head()

,match_id,city,venue,gender,teams,team,over_ball,runs_total,wicket_kind,player_out,bowling_team,batting_team
0,1,NaN,Melbourne Cricket Ground,male,"[Australia, Sri Lanka]",Australia,0.1,0,NaN,NaN,Sri Lanka,Australia
1,1,NaN,Melbourne Cricket Ground,male,"[Australia, Sri Lanka]",Australia,0.2,0,NaN,NaN,Sri Lanka,Australia
2,1,NaN,Melbourne Cricket Ground,male,"[Australia, Sri Lanka]",Australia,0.3,1,NaN,NaN,Sri Lanka,Australia
3,1,NaN,Melbourne Cricket Ground,male,"[Australia, Sri Lanka]",Australia,0.4,2,NaN,NaN,Sri Lanka,Australia
4,1,NaN,Melbourne Cricket Ground,male,"[Australia, Sri Lanka]",Australia,0.5,0,NaN,NaN,Sri Lanka,Australia


In [37]:
df_copy=df_copy.drop(['teams','team','gender'],axis=1)

In [38]:
ab=df_copy['batting_team'].unique().value_counts()
ab

Australia                  1
Sri Lanka                  1
Hong Kong                  1
Ireland                    1
Zimbabwe                   1
                          ..
Ivory Coast                1
Suriname                   1
Turks and Caicos Island    1
Timor-Leste                1
Zambia                     1
Name: count, Length: 107, dtype: int64

In [39]:
top_teams = [
    'India', 'Australia', 'England', 'Pakistan',
    'South Africa', 'New Zealand', 'West Indies',
    'Sri Lanka', 'Bangladesh', 'Afghanistan',
    'Zimbabwe', 'Ireland', 'Scotland', 
    'Netherlands', 'UAE'
]

df_copy = df_copy[
    df_copy['batting_team'].isin(top_teams) & 
    df_copy['bowling_team'].isin(top_teams)
]

In [40]:
df_copy.shape

(273888, 9)

In [41]:
import numpy as np

df_copy['city'] = np.where(
    df_copy['city'].isnull(),                             
    df_copy['venue'].str.split().apply(lambda x: x[0]),  
    df_copy['city']                                          
)

In [42]:
df_copy=df_copy.drop(['venue'],axis=1)

In [43]:
df_copy['current_score'] = df_copy.groupby('match_id')['runs_total'].cumsum()
df_copy.head()

,match_id,city,over_ball,runs_total,wicket_kind,player_out,bowling_team,batting_team,current_score
0,1,Melbourne,0.1,0,NaN,NaN,Sri Lanka,Australia,0
1,1,Melbourne,0.2,0,NaN,NaN,Sri Lanka,Australia,0
2,1,Melbourne,0.3,1,NaN,NaN,Sri Lanka,Australia,1
3,1,Melbourne,0.4,2,NaN,NaN,Sri Lanka,Australia,3
4,1,Melbourne,0.5,0,NaN,NaN,Sri Lanka,Australia,3


In [44]:
df_copy['over']=df_copy['over_ball'].apply(lambda x:str(x).split('.')[0])

In [45]:
df_copy['ball']=df_copy['over_ball'].apply(lambda x:str(x).split('.')[1])

In [46]:
df_copy['balls_bowled']=(df_copy['over'].astype('int')*6+df_copy['ball'].astype('int'))

In [47]:
df_copy['balls_left']=120-df_copy['balls_bowled']
df_copy['balls_left']=df_copy['balls_left'].apply(lambda x:0 if x<0 else x)
df_copy.head()

,match_id,city,over_ball,runs_total,wicket_kind,player_out,bowling_team,batting_team,current_score,over,ball,balls_bowled,balls_left
0,1,Melbourne,0.1,0,NaN,NaN,Sri Lanka,Australia,0,0,1,1,119
1,1,Melbourne,0.2,0,NaN,NaN,Sri Lanka,Australia,0,0,2,2,118
2,1,Melbourne,0.3,1,NaN,NaN,Sri Lanka,Australia,1,0,3,3,117
3,1,Melbourne,0.4,2,NaN,NaN,Sri Lanka,Australia,3,0,4,4,116
4,1,Melbourne,0.5,0,NaN,NaN,Sri Lanka,Australia,3,0,5,5,115


In [48]:
df_copy['is_wicket'] = df_copy['player_out'].notna().astype(int)

In [49]:
df_copy['wickets_left']=10-(df_copy.groupby('match_id')['is_wicket'].cumsum())

In [50]:
df_copy.drop(['is_wicket','player_out','wicket_kind'],axis=1)
df_copy.head()

,match_id,city,over_ball,runs_total,wicket_kind,player_out,bowling_team,batting_team,current_score,over,ball,balls_bowled,balls_left,is_wicket,wickets_left
0,1,Melbourne,0.1,0,NaN,NaN,Sri Lanka,Australia,0,0,1,1,119,0,10
1,1,Melbourne,0.2,0,NaN,NaN,Sri Lanka,Australia,0,0,2,2,118,0,10
2,1,Melbourne,0.3,1,NaN,NaN,Sri Lanka,Australia,1,0,3,3,117,0,10
3,1,Melbourne,0.4,2,NaN,NaN,Sri Lanka,Australia,3,0,4,4,116,0,10
4,1,Melbourne,0.5,0,NaN,NaN,Sri Lanka,Australia,3,0,5,5,115,0,10


In [51]:
df_copy['crr']=(df_copy['current_score']*6/df_copy['balls_bowled'])

In [52]:
df_copy=df_copy.drop(['wicket_kind','over_ball','player_out','is_wicket'],axis=1)

In [53]:
df_copy['final_score'] = df_copy.groupby('match_id')['runs_total'].transform('sum')

In [54]:
df_copy['final_score']

0         340
1         340
2         340
3         340
4         340
         ... 
758136    258
758137    258
758138    258
758139    258
758140    258
Name: final_score, Length: 273888, dtype: int64

In [55]:
df_copy=df_copy.drop(['over','ball','balls_bowled'],axis=1)
df_copy.head(20)

,match_id,city,runs_total,bowling_team,batting_team,current_score,balls_left,wickets_left,crr,final_score
0,1,Melbourne,0,Sri Lanka,Australia,0,119,10,0.000000,340
1,1,Melbourne,0,Sri Lanka,Australia,0,118,10,0.000000,340
2,1,Melbourne,1,Sri Lanka,Australia,1,117,10,2.000000,340
3,1,Melbourne,2,Sri Lanka,Australia,3,116,10,4.500000,340
4,1,Melbourne,0,Sri Lanka,Australia,3,115,10,3.600000,340
5,1,Melbourne,3,Sri Lanka,Australia,6,114,10,6.000000,340
6,1,Melbourne,0,Sri Lanka,Australia,6,113,10,5.142857,340
7,1,Melbourne,1,Sri Lanka,Australia,7,112,10,5.250000,340
8,1,Melbourne,0,Sri Lanka,Australia,7,111,10,4.666667,340
9,1,Melbourne,0,Sri Lanka,Australia,7,110,10,4.200000,340


In [56]:
df_copy['wickets_left']=df_copy['wickets_left'].apply(lambda x:0 if x<0 else x)

In [57]:
df_copy['crr'] = df_copy['crr'].round(2)

In [58]:
df_copy.head(10)

,match_id,city,runs_total,bowling_team,batting_team,current_score,balls_left,wickets_left,crr,final_score
0,1,Melbourne,0,Sri Lanka,Australia,0,119,10,0.00,340
1,1,Melbourne,0,Sri Lanka,Australia,0,118,10,0.00,340
2,1,Melbourne,1,Sri Lanka,Australia,1,117,10,2.00,340
3,1,Melbourne,2,Sri Lanka,Australia,3,116,10,4.50,340
4,1,Melbourne,0,Sri Lanka,Australia,3,115,10,3.60,340
5,1,Melbourne,3,Sri Lanka,Australia,6,114,10,6.00,340
6,1,Melbourne,0,Sri Lanka,Australia,6,113,10,5.14,340
7,1,Melbourne,1,Sri Lanka,Australia,7,112,10,5.25,340
8,1,Melbourne,0,Sri Lanka,Australia,7,111,10,4.67,340
9,1,Melbourne,0,Sri Lanka,Australia,7,110,10,4.20,340


In [60]:
df_copy.to_csv('cric_data.csv',index=False)

In [61]:
df_copy['last_five'] = (
    df_copy.groupby('match_id')['runs_total']
    .rolling(30, min_periods=1)
    .sum()
    .reset_index(level=0, drop=True)
)

In [64]:
df_copy.head(40)

,match_id,city,runs_total,bowling_team,batting_team,current_score,balls_left,wickets_left,crr,final_score,last_five
0,1,Melbourne,0,Sri Lanka,Australia,0,119,10,0.00,340,0.0
1,1,Melbourne,0,Sri Lanka,Australia,0,118,10,0.00,340,0.0
2,1,Melbourne,1,Sri Lanka,Australia,1,117,10,2.00,340,1.0
3,1,Melbourne,2,Sri Lanka,Australia,3,116,10,4.50,340,3.0
4,1,Melbourne,0,Sri Lanka,Australia,3,115,10,3.60,340,3.0
5,1,Melbourne,3,Sri Lanka,Australia,6,114,10,6.00,340,6.0
6,1,Melbourne,0,Sri Lanka,Australia,6,113,10,5.14,340,6.0
7,1,Melbourne,1,Sri Lanka,Australia,7,112,10,5.25,340,7.0
8,1,Melbourne,0,Sri Lanka,Australia,7,111,10,4.67,340,7.0
9,1,Melbourne,0,Sri Lanka,Australia,7,110,10,4.20,340,7.0


In [65]:
df_copy.to_csv('prepared_cric_data.csv',index=False)